# RoboBarista – LeRobot + RoboCasa (Google Colab)

**Ziel:** Eine existierende, auf RoboCasa trainierte **SmolVLA-Policy** in einer RoboCasa-/MuJoCo-Küche ausführen. Später prüfen wir a) wechselnde Startpositionen, b) unerwartete Positionsänderungen während einer Bewegung und c) messbare Positions- und Winkelfehler.

**Wichtig:** Dieses Notebook richtet zuerst die **offizielle** Integration ein und führt Basistests aus. Es behauptet **noch nicht**, dass die Policy verschobene Gegenstände präzise greift oder eine ganze Kaffeeabfolge zuverlässig ausführen kann. Für eine echte mehrstufige Manipulationssequenz und kontrollierte Eingriffe während der Bewegung folgt nach dem erfolgreichen Basistest ein eigener Test-Harness. Wir programmieren dabei **keine Greif-Policy** selbst.

**Versionen:** LeRobot `v0.6.1` (Python ≥ 3.12), robosuite CI-Commit `aaa8b9b214ce` (aus der offiziellen LeRobot-RoboCasa-Docker-Konfiguration), dein RoboCasa-Branch `barista-dev` (RoboCasa 1.0.1). Reproduzierbare Auswahl, aber Zusammenspiel auf Colab **erst im Test zu verifizieren**.

**Umgebung:** In Colab unter *Laufzeit → Laufzeittyp ändern* NVIDIA T4 wählen. Zellen nacheinander ausführen. Die Installation erfolgt **nur** unter `/content/robobarista-py312`; deine vorherige Python-3.11-Umgebung bleibt erhalten. Colab-Dateien sind temporär.

Offizielle Anleitung: https://huggingface.co/docs/lerobot/v0.6.1/robocasa (alternativ https://huggingface.co/docs/lerobot/main/robocasa). Policy: https://huggingface.co/lerobot/smolvla_robocasa.

## 1. GPU, RAM und freien Speicher prüfen

Der Test stoppt, falls keine NVIDIA-GPU oder weniger als 20 GiB freier Speicher zur Verfügung stehen.

In [ ]:
import os, shutil, subprocess
from pathlib import Path

assert shutil.which("nvidia-smi"), "In Colab zuerst T4-GPU aktivieren: Laufzeit → Laufzeittyp ändern."
subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], check=True)
print("Freier Speicher:", round(shutil.disk_usage("/content").free / 1024**3, 1), "GiB")
assert shutil.disk_usage("/content").free >= 20 * 1024**3, "Mindestens 20 GiB frei empfohlen. Keine Installation gestartet."
print("Aktuelles Notebook-Python:", subprocess.check_output(["python", "--version"], text=True).strip())

## 2. Projekt und offizielle Bibliotheken holen

Dein RoboCasa-Fork wird von `barista-dev` geladen, LeRobot vom Tag `v0.6.1`. **robosuite wird auf den von LeRobots offizieller RoboCasa-CI benutzten Commit gepinnt** (nicht auf `v1.5.2`, dem `load_model_on_init` fehlt). Bereits vorhandene Verzeichnisse werden erhalten, sofern das Arbeitsverzeichnis sauber ist. Nur robosuite wird bei Bedarf auf den getesteten Stand umgestellt.


In [ ]:
import subprocess
from pathlib import Path

repos = [
    ("https://github.com/Coffee-Tobi/robocasa.git", "/content/robocasa", "barista-dev"),
    ("https://github.com/huggingface/lerobot.git", "/content/lerobot", "v0.6.1"),
    ("https://github.com/ARISE-Initiative/robosuite.git", "/content/robosuite", "master"),
]
for url, folder, ref in repos:
    if not (Path(folder) / ".git").exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", ref, url, folder], check=True)
    else:
        print("Bereits vorhanden:", folder)
    print(folder, subprocess.check_output(["git", "-C", folder, "rev-parse", "--short", "HEAD"], text=True).strip())

# Exakter Commit aus huggingface/lerobot docker/Dockerfile.benchmark.robocasa (v0.6.1).
# Die robosuite-Version v1.5.2 kennt 'load_model_on_init' noch nicht.
ROBOSUITE_SHA = "aaa8b9b214ce8e77e82926d677b4d61d55e577ab"
rs = "/content/robosuite"
working_changes = subprocess.check_output(["git", "-C", rs, "status", "--porcelain"], text=True).strip()
if working_changes:
    raise RuntimeError("Nicht gespeicherte Änderungen in robosuite: kein Checkout durchgeführt.\n" + working_changes)
current_sha = subprocess.check_output(["git", "-C", rs, "rev-parse", "HEAD"], text=True).strip()
if current_sha != ROBOSUITE_SHA:
    subprocess.run(["git", "-C", rs, "fetch", "--depth", "1", "origin", ROBOSUITE_SHA], check=True)
    subprocess.run(["git", "-C", rs, "checkout", "--detach", "FETCH_HEAD"], check=True)
actual_sha = subprocess.check_output(["git", "-C", rs, "rev-parse", "HEAD"], text=True).strip()
assert actual_sha == ROBOSUITE_SHA, (actual_sha, ROBOSUITE_SHA)
print("robosuite CI-Commit aktiv:", actual_sha[:12])


## 3. Saubere Python-3.12-Umgebung anlegen

Die bisherige Python-3.11-Umgebung unter `/content/robocasa-env` bleibt bestehen; LeRobot 0.6.1 benötigt Python 3.12+. Die Installation dieser Umgebung muss nach einem vollständigen Colab-Laufzeitreset wiederholt werden.

In [ ]:
from pathlib import Path
import subprocess

MICRO = "/content/bin/bin/micromamba"
if not Path(MICRO).is_file():
    subprocess.run("mkdir -p /content/bin && curl -Ls https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xvj -C /content/bin bin/micromamba", shell=True, check=True)
PY_ENV = "/content/robobarista-py312"
if not Path(PY_ENV, "bin/python").is_file():
    subprocess.run([MICRO, "create", "-y", "-p", PY_ENV, "-c", "conda-forge", "python=3.12", "pip"], check=True)
PY = f"{PY_ENV}/bin/python"
print(subprocess.check_output([PY, "--version"], text=True).strip())

## 4. Offizielle Pakete installieren

**Es werden mehrere GB Pakete heruntergeladen.** Dieser Schritt kann in der kostenlosen Colab-Laufzeit einige Zeit dauern. Wir nutzen kein MimicGen und verändern Colabs Python 3.13 nicht.

Die RoboCasa-Installation erfolgt mit `--no-deps`, weil dessen `setup.py` LeRobot 0.3.3 erzwingt, obwohl wir LeRobot 0.6.1 brauchen. Die benötigten Runtime-Pakete installieren wir gezielt nach der offiziellen LeRobot-RoboCasa-Anleitung. Fehler bitte vollständig teilen, statt auf Verdacht weitere Pakete zu installieren.

In [ ]:
import os, subprocess

env = os.environ.copy()
env["PIP_NO_CACHE_DIR"] = "1"
env["PIP_DISABLE_PIP_VERSION_CHECK"] = "1"

def pip_install(*args):
    print("\n>>> pip install", " ".join(args), flush=True)
    subprocess.run([PY, "-m", "pip", "install", "--no-cache-dir", *args], env=env, check=True)

pip_install("-e", "/content/lerobot[smolvla,dataset,evaluation]")
pip_install("-e", "/content/robosuite")
pip_install(
    "numpy==2.2.5", "numba==0.61.2", "scipy==1.15.3", "mujoco==3.3.1",
    "pygame", "Pillow", "opencv-python", "pyyaml", "pynput", "tqdm",
    "termcolor", "imageio", "h5py", "lxml", "hidapi", "tianshou==0.4.10",
    "gymnasium",
)
pip_install("--no-deps", "-e", "/content/robocasa")
print("\nInstallation abgeschlossen. Als nächstes werden Imports und die GPU geprüft.")

## 5. GPU und Python-Imports in genau dieser Umgebung testen

Prüft ebenfalls ausdrücklich die RoboCasa/robosuite-Schnittstelle `load_model_on_init` gegen die in LeRobot dokumentierte Version. Erst nach erfolgreichem Test mit Assets/Policy fortfahren.


In [ ]:
import os, subprocess

env = os.environ.copy()
env["MUJOCO_GL"] = "egl"
env["PYOPENGL_PLATFORM"] = "egl"
env["OPENBLAS_NUM_THREADS"] = "1"

check = """import sys
import importlib
from importlib.metadata import version, PackageNotFoundError
import torch

for module_name, dist_name in [
    ("torch", "torch"),
    ("mujoco", "mujoco"),
    ("robosuite", "robosuite"),
    ("robocasa", "robocasa"),
    ("lerobot", "lerobot"),
]:
    importlib.import_module(module_name)
    try:
        installed_version = version(dist_name)
    except PackageNotFoundError:
        installed_version = "importierbar (keine Paket-Metadaten)"
    print(f"{module_name}: OK ({installed_version})", flush=True)

from robosuite.environments.manipulation.manipulation_env import ManipulationEnv\nimport inspect\nassert "load_model_on_init" in inspect.signature(ManipulationEnv.__init__).parameters, "robosuite falscher Commit: load_model_on_init fehlt"\nprint("robosuite-Schnittstelle load_model_on_init: OK", flush=True)\nprint("Python:", sys.version.split()[0], flush=True)
print("CUDA:", torch.cuda.is_available(), flush=True)
assert torch.cuda.is_available(), "PyTorch erkennt die Colab-GPU nicht"
print("GPU:", torch.cuda.get_device_name(0), flush=True)
"""

subprocess.run([PY, "-u", "-c", check], env=env, check=True)


## 6. RoboCasa-Assets herunterladen (nur leichter Satz)

Dieser Schritt erstellt die private Konfiguration **nur wenn sie fehlt**; vorhandene Einstellungen werden nicht überschrieben. Danach lädt er nur Texturen und Lightwheel-Assets. **Falls die Zelle erneut gestartet wird, können Assets erneut heruntergeladen werden.**


In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

# RoboCasa setup_macros.py fragt interaktiv nach, falls die Datei existiert.
# In Colab stattdessen idempotent einrichten und vorhandene Einstellungen erhalten.
macros = Path("/content/robocasa/robocasa/macros.py")
macros_private = macros.with_name("macros_private.py")
if not macros.exists():
    raise FileNotFoundError(f"RoboCasa-Konfiguration nicht gefunden: {macros}")
if macros_private.exists():
    print("Private RoboCasa-Konfiguration bereits vorhanden; unverändert gelassen.", flush=True)
else:
    shutil.copy2(macros, macros_private)
    print(f"Private RoboCasa-Konfiguration erstellt: {macros_private}", flush=True)

env = os.environ.copy()
env.update({"MUJOCO_GL": "egl", "PYOPENGL_PLATFORM": "egl"})

# Nur den leichten offiziellen Assetsatz holen, nicht Objaverse (~30 GB).
# Der Downloader erwartet einmal 'y'; subprocess liefert das gezielt.
cmd = [
    PY, "-m", "robocasa.scripts.download_kitchen_assets",
    "--type", "tex", "tex_generative", "fixtures_lw", "objs_lw",
]
print("Starte RoboCasa-Asset-Download (Lightwheel + Texturen) ...", flush=True)
subprocess.run(cmd, input="y\n", text=True, env=env, check=True)
print("Asset-Download beendet.", flush=True)


### 6b. Einrichtungsmodelle prüfen und fehlendes Paket reparieren

**Warum nötig?** Der offizielle RoboCasa-Downloader kann nach einem Netzwerkfehler enden, ohne eine fehlende Datei zuverlässig zu melden. Der Basistest benötigt unter anderem `fixtures/windows/Window052/model.xml`. Wir prüfen zuerst diese Datei und laden **nur wenn nötig** das etwa **505 MiB** große offizielle Lightwheel-Fixtures-Paket von einem dokumentierten Hugging-Face-Spiegel. Die veröffentlichte SHA-256-Prüfsumme wird vor dem Entpacken geprüft. Bestehende andere Assets und die LeRobot-Installation bleiben unberührt.

Quellen: https://huggingface.co/datasets/IIFAN/robocasa365-lightwheel-fixtures (Mirror mit SHA256 und Archivpfad) und https://github.com/robocasa/robocasa/blob/main/robocasa/scripts/download_kitchen_assets.py (ursprünglicher Downloadmechanismus).


In [ ]:
import subprocess

# Führt den Download in der installierten Python-3.12-Umgebung aus;
# Notebook-Python 3.13 benötigt dafür keine zusätzlichen Pakete.
repair_script = "from pathlib import Path\nfrom zipfile import ZipFile\nfrom hashlib import sha256\nfrom shutil import disk_usage\nfrom huggingface_hub import hf_hub_download\n\nassets_dir = Path(\"/content/robocasa/robocasa/models/assets\")\nrequired = assets_dir / \"fixtures/windows/Window052/model.xml\"\nprint(\"Benötigte Fixture-Datei:\", required)\nprint(\"Vorhanden:\", required.is_file())\n\nif not required.is_file():\n    # Archiv: 529,745,465 Bytes (rund 505 MiB); entpackt etwa 1.44 GB.\n    assert disk_usage(\"/content\").free > 3 * 1024**3, \"Zu wenig freier Speicher für Fixtures-Archiv und Entpacken.\"\n    archive = Path(hf_hub_download(\n        repo_id=\"IIFAN/robocasa365-lightwheel-fixtures\",\n        filename=\"fixtures_lightwheel.zip\",\n        repo_type=\"dataset\",\n        local_dir=\"/content/robobarista_fixture_download\",\n    ))\n    print(\"Archiv geladen:\", archive, \"Bytes:\", archive.stat().st_size)\n    expected_size = 529745465\n    expected_sha = \"62c1c87776edda9b7c42dda7b00781a1e5a7f90acc3e49d06c39c9545cbd6c3e\"\n    assert archive.stat().st_size == expected_size, \"Archivgröße stimmt nicht mit dokumentierter Referenz überein.\"\n    checksum = sha256()\n    with archive.open(\"rb\") as f:\n        for chunk in iter(lambda: f.read(8 * 1024 * 1024), b\"\"):\n            checksum.update(chunk)\n    assert checksum.hexdigest() == expected_sha, \"SHA256-Prüfsumme falsch; nichts wird entpackt.\"\n\n    with ZipFile(archive) as z:\n        names = z.namelist()\n        target = \"fixtures/windows/Window052/model.xml\"\n        assert target in names, (\n            \"Das Archiv enthält Window052/model.xml nicht. \"\n            \"Bitte die Ausgabe teilen, statt weitere Pakete auf Verdacht zu installieren.\"\n        )\n        print(\"Prüfsumme OK, Window052 im Archiv gefunden. Entpacke nur Fixture-Assets ...\", flush=True)\n        z.extractall(assets_dir)\n\nassert required.is_file(), \"Window052/model.xml nach Asset-Reparatur nicht vorhanden.\"\nprint(\"✅ Window052/model.xml gefunden. Basistest kann erneut gestartet werden.\")\n"
subprocess.run(
    ["/content/robobarista-py312/bin/python", "-u", "-c", repair_script],
    check=True,
)


### 6c. Lightwheel-Objekte vollständig einrichten (fehlendes Stool020)

Die Umgebung benötigt nicht nur Fixtures wie Fenster und Kühlschränke, sondern auch das separate `objs_lw`-Paket. Fehlt `objects/lightwheel/stool/Stool020/model.xml`, ist das Objektpaket unvollständig. Wir laden **nur dieses Paket** vom dokumentierten [Lightwheel-Objektspiegel](https://huggingface.co/datasets/IIFAN/robocasa365-lightwheel-objects) (~792 MB ZIP, ~1.45 GB entpackt). Vor dem Entpacken prüfen wir Größe, SHA-256 und den geforderten Archivpfad. Bereits vorhandene andere Assetpakete bleiben bestehen.

Nach diesem Schritt erneut den offiziellen `CloseFridge`-Basistest ausführen.

In [ ]:
import subprocess

script = "from pathlib import Path\nfrom hashlib import sha256\nfrom shutil import disk_usage\nfrom zipfile import ZipFile\nfrom huggingface_hub import hf_hub_download\n\nroot = Path(\"/content/robocasa/robocasa/models/assets/objects\")\ntarget_name = \"lightwheel/stool/Stool020/model.xml\"\nrequired = root / target_name\n\nif required.is_file():\n    print(\"✅ Lightwheel-Hocker bereits vorhanden:\", required, flush=True)\nelse:\n    assert disk_usage(\"/content\").free > 4 * 1024**3, \"Zu wenig freier Speicher (mindestens 4 GiB empfohlen).\"\n    print(\"Lade das vollständige Lightwheel-Objektpaket ...\", flush=True)\n    archive = Path(hf_hub_download(\n        repo_id=\"IIFAN/robocasa365-lightwheel-objects\",\n        filename=\"objects_lightwheel.zip\",\n        repo_type=\"dataset\",\n        local_dir=\"/content/robobarista_objects_download\",\n    ))\n    expected_bytes = 791939656\n    assert archive.stat().st_size == expected_bytes, (\n        f\"Archivgröße stimmt nicht: {archive.stat().st_size} statt {expected_bytes}\"\n    )\n    checksum = sha256()\n    with archive.open(\"rb\") as f:\n        for chunk in iter(lambda: f.read(8 * 1024 * 1024), b\"\"):\n            checksum.update(chunk)\n    expected_sha = \"a1db739b0e80fc4d56d4be27d89dba3aa60000e9648c5607874700e0f14b7955\"\n    assert checksum.hexdigest() == expected_sha, \"SHA-256-Prüfsumme stimmt nicht; keine Entpackung.\"\n\n    with ZipFile(archive) as z:\n        names = z.namelist()\n        assert target_name in names, (\n            f\"Stool020 fehlt auch im Archiv! Beispielpfade: \"\n            f\"{[n for n in names if 'Stool020' in n][:8]}\"\n        )\n        # Community-Archiv: Pfade vor dem Entpacken gegen Path-Traversal prüfen.\n        root_abs = root.resolve()\n        assert all((root_abs / n).resolve().is_relative_to(root_abs) for n in names), (\n            \"Archiv enthält unerlaubte Pfade, nichts entpackt.\"\n        )\n        print(\"SHA-256 OK; Stool020 im Archiv gefunden. Entpacke Objekte ...\", flush=True)\n        z.extractall(root)\n    assert required.is_file(), \"Stool020 fehlt nach dem Entpacken.\"\n    print(\"✅ Lightwheel-Objektpaket bereit:\", required, flush=True)\n\nprint(\"Asset-Verzeichnisprüfung:\")\nassets = root.parent\nfor folder in (\"fixtures\", \"objects/lightwheel\", \"textures\", \"generative_textures\"):\n    path = assets / folder\n    print(f\"{folder}: {'vorhanden' if path.is_dir() else 'FEHLT'}\")\n"
subprocess.run(
    ["/content/robobarista-py312/bin/python", "-u", "-c", script],
    check=True,
)


### 6d. Generierte Wandtexturen reparieren (fehlendes tex027.png)

Beim Erzeugen der RoboCasa-Küche wurde `generative_textures/wall/tex027.png` nicht gefunden. Das bedeutet, dass **das separate generative Texturenpaket** noch fehlt oder unvollständig ist. Wir verwenden hierfür das [Asset-Repository der RoboCasa-Organisation](https://huggingface.co/datasets/robocasa/robocasa-assets/tree/main), und pinnen seinen SHA-verifizierten Archiv-Stand (`1b92c3d`). Das Archiv hat **1.18 GB**, der Download kann einige Minuten dauern.

Der Reparaturcode vergleicht Dateigröße und SHA-256 mit [dem veröffentlichten Repository-Commit](https://huggingface.co/datasets/robocasa/robocasa-assets/commit/1b92c3d02ca4354984fec961357db0bff7b32166), prüft dann, ob `tex027.png` wirklich im Archiv liegt, und entpackt die vollständigen Texturen an die von RoboCasa erwartete Stelle. Wenn die Datei schon existiert, passiert nichts. Bereits heruntergeladene Fixtures, Objekte und Modellgewichte werden nicht verändert.

In [ ]:
import subprocess

# Separates Python 3.12, in dem huggingface_hub bereits installiert ist.
code = "from pathlib import Path\nfrom hashlib import sha256\nfrom shutil import disk_usage\nfrom zipfile import ZipFile\nfrom huggingface_hub import hf_hub_download\n\nassets = Path(\"/content/robocasa/robocasa/models/assets\")\ntarget = assets / \"generative_textures/wall/tex027.png\"\nif target.is_file():\n    print(\"✅ Generierte Wandtextur bereits vorhanden:\", target)\nelse:\n    assert disk_usage(\"/content\").free > 5 * 1024**3, \"Weniger als 5 GiB freier Speicher; bitte zuerst Speicher prüfen.\"\n    print(\"Lade offizielles RoboCasa-Texturarchiv (~1.18 GB) ...\", flush=True)\n    archive = Path(hf_hub_download(\n        repo_id=\"robocasa/robocasa-assets\",\n        repo_type=\"dataset\",\n        filename=\"generative_textures.zip\",\n        revision=\"1b92c3d02ca4354984fec961357db0bff7b32166\",\n        local_dir=\"/content/robobarista_generative_download\",\n    ))\n    expected_size = 1184675536\n    expected_digest = \"bc05095af306222d9a799a9b0b5b16662aa546998622ee8fd3072566ea4c31ec\"\n    assert archive.stat().st_size == expected_size, (\n        f\"Falsche Dateigröße: {archive.stat().st_size} statt {expected_size}\"\n    )\n    digest = sha256()\n    with archive.open(\"rb\") as f:\n        for chunk in iter(lambda: f.read(8 * 1024 * 1024), b\"\"):\n            digest.update(chunk)\n    assert digest.hexdigest() == expected_digest, (\n        \"SHA-256 der Archivdatei weicht vom offiziellen Commit ab. Nicht entpackt.\"\n    )\n\n    with ZipFile(archive) as z:\n        files = set(z.namelist())\n        rel_a = \"generative_textures/wall/tex027.png\"\n        rel_b = \"wall/tex027.png\"\n        if rel_a in files:\n            dest = assets\n        elif rel_b in files:\n            dest = assets / \"generative_textures\"\n        else:\n            matches = [p for p in files if p.endswith(\"/wall/tex027.png\")][:10]\n            raise RuntimeError(\n                f\"tex027.png nicht am erwarteten Archivpfad; Varianten: {matches}. \"\n                \"Bitte Ausgabe teilen; nichts entpackt.\"\n            )\n\n        root_abs = dest.resolve()\n        for item in z.infolist():\n            candidate = (root_abs / item.filename).resolve()\n            if not candidate.is_relative_to(root_abs):\n                raise RuntimeError(f\"Unsicherer ZIP-Pfad: {item.filename}\")\n        print(\"✅ Integrität geprüft, tex027.png gefunden. Entpacke Texturen ...\", flush=True)\n        z.extractall(dest)\n\n    assert target.is_file(), \"Textur fehlt nach dem Entpacken: \" + str(target)\n    print(\"✅ Generative Texturen vollständig bereit:\", target)\n\n# Auch das Basispaket prüfen: Es wird bei MuJoCo-Szenen benötigt.\nbase_texture = assets / \"textures/flat/white.png\"\nprint(\"Basistextrur vorhanden:\", base_texture.is_file(), base_texture)\n"
subprocess.run(
    ["/content/robobarista-py312/bin/python", "-u", "-c", code],
    check=True,
)


### 6e. Normale Texturen reparieren und alle Assets prüfen

Beim Start der MuJoCo-Küche wurde `textures/tiles/orange_square_tiles.png` nicht gefunden. Das ist das **Basistexturpaket** (`tex`) und unterscheidet sich von den generierten Wandtexturen (`tex_generative`). Wir verwenden das offizielle [RoboCasa-Asset-Repository](https://huggingface.co/datasets/robocasa/robocasa-assets/tree/1b92c3d02ca4354984fec961357db0bff7b32166) mit fester Revision. Die ZIP-Datei `textures.zip` ist 543.180.421 Bytes groß; SHA-256 gemäß [RoboCasa-Commit](https://huggingface.co/datasets/robocasa/robocasa-assets/commit/1b92c3d02ca4354984fec961357db0bff7b32166): `ba3a4dcef96e199cbd601c7cdb7eb46077fb6672611b7ac74c25a191bf0be392`.

Nach dem Download prüfen wir **alle vier bisher benötigten Assetgruppen**: Basistexturen, generierte Texturen, Fenster-Fixtures und Lightwheel-Objekte. Wenn etwas fehlt, stoppt die Zelle **vor der Policy-Evaluation** mit klarer Diagnose. Bereits vorhandene Modelle und Software bleiben unverändert.

In [ ]:
import subprocess

code = "from pathlib import Path\nfrom hashlib import sha256\nfrom shutil import disk_usage\nfrom zipfile import ZipFile\nfrom huggingface_hub import hf_hub_download\n\nassets = Path(\"/content/robocasa/robocasa/models/assets\")\ntarget_rel = \"textures/tiles/orange_square_tiles.png\"\ntarget = assets / target_rel\n\nif not target.is_file():\n    assert disk_usage(\"/content\").free > 3 * 1024**3, \"Weniger als 3 GiB freier Speicher: Download zunächst stoppen.\"\n    print(\"Basistexturen fehlen; lade offizielles textures.zip (~543 MB) ...\", flush=True)\n    archive = Path(hf_hub_download(\n        repo_id=\"robocasa/robocasa-assets\",\n        repo_type=\"dataset\",\n        filename=\"textures.zip\",\n        revision=\"1b92c3d02ca4354984fec961357db0bff7b32166\",\n        local_dir=\"/content/robobarista_base_textures\",\n    ))\n    expected_size = 543180421\n    expected_digest = \"ba3a4dcef96e199cbd601c7cdb7eb46077fb6672611b7ac74c25a191bf0be392\"\n    if archive.stat().st_size != expected_size:\n        raise RuntimeError(f\"Falsche Archivgröße: {archive.stat().st_size} statt {expected_size}\")\n    digest = sha256()\n    with archive.open(\"rb\") as handle:\n        for part in iter(lambda: handle.read(8*1024*1024), b\"\"):\n            digest.update(part)\n    if digest.hexdigest() != expected_digest:\n        raise RuntimeError(f\"Prüfsumme falsch: {digest.hexdigest()}\")\n\n    with ZipFile(archive) as z:\n        names = set(z.namelist())\n        # Der offizielle Downloader entpackt nach models/assets; damit ist\n        # die übliche ZIP-Struktur textures/... erwartet.\n        if target_rel in names:\n            dest = assets\n        elif \"tiles/orange_square_tiles.png\" in names:\n            dest = assets / \"textures\"\n        else:\n            hits = [x for x in names if x.endswith(\"orange_square_tiles.png\")]\n            raise RuntimeError(\n                f\"Textur nicht am erwarteten Archivpfad. Mögliche Pfade: {hits[:8]}\"\n            )\n        dest_abs = dest.resolve()\n        for item in z.infolist():\n            candidate = (dest_abs / item.filename).resolve()\n            if not candidate.is_relative_to(dest_abs):\n                raise RuntimeError(f\"Unsicherer Archivpfad: {item.filename}\")\n        print(\"Prüfsumme OK. Entpacke das vollständige Texturpaket ...\", flush=True)\n        z.extractall(dest)\n\nrequired = {\n    \"Basistexturen\": \"textures/tiles/orange_square_tiles.png\",\n    \"Generierte Texturen\": \"generative_textures/wall/tex027.png\",\n    \"Fenster-Fixture\": \"fixtures/windows/Window052/model.xml\",\n    \"Lightwheel-Objekte\": \"objects/lightwheel/stool/Stool020/model.xml\",\n}\nprint(\"\\nAsset-Prüfung nach Installation:\", flush=True)\nmissing = []\nfor name, rel in required.items():\n    found = (assets / rel).is_file()\n    print(f\"{'✅' if found else '❌'} {name}: {rel}\", flush=True)\n    if not found:\n        missing.append(rel)\nif missing:\n    raise RuntimeError(\n        \"Folgende Beispiel-Assets fehlen noch: \" + \", \".join(missing)\n        + \". Bitte NICHT direkt erneut evaluate() ausführen.\"\n    )\nprint(\"\\n✅ Alle vier bisher benötigten Assetgruppen sind vorhanden.\", flush=True)\n"
subprocess.run(
    ["/content/robobarista-py312/bin/python", "-u", "-c", code],
    check=True,
)


## 7. Vortrainiertes SmolVLA-RoboCasa-Modell herunterladen

Der eigentliche Checkpoint ist etwa **907 MB** groß (ohne mögliche zusätzliche Hugging-Face-Backbone-Dateien). Es wird nichts trainiert. Downloads liegen standardmäßig im temporären Hugging-Face-Cache. Ein späterer Evaluation-Aufruf lädt bei Bedarf weitere notwendige Modell-Dateien.

In [ ]:
import subprocess, os

env=os.environ.copy(); env["HF_HOME"]="/content/robobarista_hf_cache"
subprocess.run([PY, "-c", "from huggingface_hub import snapshot_download; p=snapshot_download('lerobot/smolvla_robocasa'); print('Checkpoint:',p)"], env=env, check=True)

## 8. Offizieller Basistest: eine RoboCasa-Aufgabe

**Erst dies bestätigt**, ob die Policy auf unserer T4 tatsächlich in MuJoCo läuft. Wir verwenden die **unveränderten offiziellen** `lerobot-eval`-Parameter einschließlich Kamera-Umbenennung. Zunächst **eine** Episode, um nicht unnötig GPU-Kontingent zu verbrauchen. Dies ist keine Präzisions- oder Live-Störungsmessung.

In [ ]:
import os, subprocess
from pathlib import Path

Path("/content/robobarista_runs").mkdir(exist_ok=True)
eval_env=os.environ.copy()
eval_env.update({
    "MUJOCO_GL":"egl", "PYOPENGL_PLATFORM":"egl",
    "OPENBLAS_NUM_THREADS":"1", "NUMBA_NUM_THREADS":"1",
    "HF_HOME":"/content/robobarista_hf_cache",
})
RENAME_MAP='{"observation.images.robot0_agentview_left": "observation.images.camera1", "observation.images.robot0_eye_in_hand": "observation.images.camera2", "observation.images.robot0_agentview_right": "observation.images.camera3"}'

def evaluate(task, n_episodes=1):
    cmd=[
        f"{PY_ENV}/bin/lerobot-eval",
        "--policy.path=lerobot/smolvla_robocasa",
        "--env.type=robocasa", f"--env.task={task}",
        "--eval.batch_size=1", f"--eval.n_episodes={n_episodes}",
        "--eval.use_async_envs=false", "--policy.device=cuda",
        f"--rename_map={RENAME_MAP}",
    ]
    print("Starte offiziellen Test:",task, flush=True)
    log_path=f"/content/robobarista_runs/{task}_eval.log"
    with open(log_path,"w") as log:
        proc=subprocess.Popen(cmd,env=eval_env,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1,cwd="/content/lerobot")
        for line in proc.stdout:
            print(line,end=""); log.write(line);log.flush()
        ret=proc.wait()
    if ret:
        raise RuntimeError(f"Evaluation fehlgeschlagen (Code {ret}). Log: {log_path}")
    print("Basistest beendet. Log:",log_path)

evaluate("CloseFridge",n_episodes=1)

## 9. Als nächstes: Pick-and-Place mit variierenden Startpositionen

**Nur ausführen, wenn der Basistest oben funktioniert hat.** Die RoboCasa-Aufgabe `PickPlaceCounterToStove` ist im RoboCasa-Datensatzregister vorhanden. `n_episodes=3` führt unterschiedliche Episoden/Resets aus – die tatsächliche Variabilität der Gegenstandspositionen prüfen wir anschließend. Dies sind **separate** Episoden, noch keine verkettete Kaffee-Zubereitungssequenz.

Falls Lightwheel für eine Objektauswahl nicht ausreicht, kann die Umgebung scheitern. Dann bitte **keine 30 GB Assets blind installieren**, sondern zunächst das Fehlerprotokoll teilen.

In [ ]:
evaluate("PickPlaceCounterToStove",n_episodes=3)

## 10. Experimentplan nach erfolgreichem Basistest

Wir ergänzen **nur einen Versuchsaufbau**, keine neue Roboterbewegungslogik:

**A – Randomisierte Startpositionen:** mehrere Seeds, zufällig verschobene (physikalisch gültige) Tassen, Kännchen und Siebträger; Erfolgsquote pro Objekt erfassen.

**B – Störung im laufenden Ablauf:** Objekt in einem definierten Zeitschritt versetzen (oder durch einen physikalischen Stoß bewegen); die Policy erhält anschließend neue Kamerabilder. Wir prüfen, ob sie vor dem Greifen tatsächlich korrigiert. **Das ist mit diesem Notebook noch nicht implementiert.**

**C – Präzision:** Endposition und Orientierung gegen Sollziel messen, Kollisions- und Greiffehler gesondert protokollieren. Wir legen Toleranzen pro Aufgabe fest, statt pauschal „sehr präzise“ zu behaupten.

**D – Mehrere Objekte in Folge:** zunächst getrennte offizielle RoboCasa-Aufgaben mit SmolVLA testen; danach über eine Aufgabensteuerung mit Zustandsprüfung in **einer** Szene verketten. Die bestehende Policy ersetzt keine Zustandsmaschine und keine kraftsensitive Verriegelungsregelung.

**Bei Problemen:** Genau die fehlschlagende Zelle und ihre Ausgabe teilen. Keine unkontrollierten Downgrades. Ergebnisse/Logs können aus `/content/robobarista_runs` heruntergeladen oder nach Google Drive kopiert werden.